Prototype CNN
Sahil Mulki

In [ ]:
%pip install -q kaggle python-dotenv scikit-learn

In [ ]:
import os
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
)

In [ ]:
SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

DATA_DIR = "brain_mri_dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "Training")
TEST_DIR = os.path.join(DATA_DIR, "Testing")

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
VAL_FRACTION = 0.15
PATIENCE = 6         

In [ ]:
try:
    # training on Colab
    from google.colab import userdata  
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Loaded Kaggle credentials from Colab secrets.")
except ImportError:
    # running locally instead
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=os.path.join("..", ".env"))
    load_dotenv()
    print("Loaded Kaggle credentials from .env.")

# Download (skipped if the data is already present)
if os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR):
    print("Dataset already present. Skipping download.")
else:
    import kaggle
    kaggle.api.authenticate()
    print("Downloading dataset from Kaggle (this may take a few minutes)...")
    kaggle.api.dataset_download_files(
        "masoudnickparvar/brain-tumor-mri-dataset",
        path=DATA_DIR,
        unzip=True,
    )
    print("Done.")

In [ ]:
# Data Preprocessing
# Crop the margins
def crop_brain_region(pil_img):
    gray = np.array(pil_img.convert("L"))
    thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return pil_img
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    return pil_img.crop((x, y, x + w, y + h))

In [ ]:
# Grayscale normalization to [-1, 1].
# Fixed 0.5/0.5
NORM_MEAN, NORM_STD = (0.5,), (0.5,)

# Training transform: crop + resize
# Then data augmentation
train_transform = transforms.Compose([
    transforms.Lambda(crop_brain_region),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# Evaluation transform
# Same preprocessing as above but nO augmentation for val or test
eval_transform = transforms.Compose([
    transforms.Lambda(crop_brain_region),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])


train_view = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_view = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
class_names = train_view.classes

n_total = len(train_view)

# Fixed seed index split so an image is either train or val
perm = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED)).tolist()
n_val = int(n_total * VAL_FRACTION)
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_ds = Subset(train_view, train_idx)
val_ds = Subset(val_view, val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train: {len(train_ds)}")
print("Val: {len(val_ds)}")
print("Test: {len(train_ds)}")

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=4, in_channels=1):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                # Add BatchNorm to stabilize and speed up training
                nn.BatchNorm2d(out_c), 
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
        
        self.conv = nn.Sequential(
            block(in_channels, 32),   # 256 -> 128
            block(32, 64),            # 128 -> 64
            block(64, 128),           # 64  -> 32
            block(128, 128),          # 32  -> 16
        )
        # Compute the flattened size once so the Linear layer matches IMG_SIZE.
        with torch.no_grad():
            n_flat = self.conv(torch.zeros(1, in_channels, IMG_SIZE, IMG_SIZE)).flatten(1).shape[1]
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),           
            nn.Linear(n_flat, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


model = CNN(num_classes=len(class_names)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

# This is something to keep an eye on because the number of params is pretty high
print(f"Total parameters: {n_params:,}")

In [ ]:
# To maximize recall: weight the loss toward the tumor classes: missing a tumor should hurt more than a false alarm.

class_weights = torch.tensor(
    [1.0 if name == "notumor" else 2.0 for name in class_names], device=DEVICE
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# LR scheduler driven by val recall.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

BEST_PATH = "best_cnn.pt"


def evaluate(loader):
    """Return (accuracy, macro_recall, preds, targets) over a loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            preds.extend(model(imgs).argmax(1).cpu().tolist())
            targets.extend(labels.tolist())
    acc = float(np.mean(np.array(preds) == np.array(targets)))
    macro_recall = recall_score(targets, preds, average="macro", zero_division=0)
    return acc, macro_recall, preds, targets


history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_recall": []}
best_val_recall = 0.0
epochs_without_improvement = 0

# Model training
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    train_correct = train_total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    # validate to optimize for recall
    val_acc, val_recall, _, _ = evaluate(val_loader)
    scheduler.step(val_recall)

    history["train_loss"].append(epoch_loss / len(train_loader))
    history["train_acc"].append(train_correct / train_total)
    history["val_acc"].append(val_acc)
    history["val_recall"].append(val_recall)
    print(f"Epoch {epoch}/{EPOCHS} | train loss {history['train_loss'][-1]:.4f} | "
          f"train acc {history['train_acc'][-1]:.4f} | val acc {val_acc:.4f} | "
          f"val recall {val_recall:.4f}")

    # checkpoint and early stopping on validation macro-recall
    if val_recall > best_val_recall:
        best_val_recall = val_recall
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val-recall gain for {PATIENCE} epochs).")
            break

# Restore the weights with the best validation recall.
model.load_state_dict(torch.load(BEST_PATH))
print(f"\nBest validation macro-recall: {best_val_recall:.4f}")

In [ ]:
# Plot training loss, accuracy, and recall
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(history["train_loss"]); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
ax[2].plot(history["val_recall"]); ax[2].set_title("Validation macro-recall"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
# Eval on test set
test_acc, test_recall, preds, targets = evaluate(test_loader)
preds, targets = np.array(preds), np.array(targets)

print(f"Test accuracy: {test_acc:.4f} | test macro-recall: {test_recall:.4f}\n")
print(classification_report(targets, preds, target_names=class_names, digits=3))

# Tumor miss rate: true tumor scans predicted as "notumor" -- the most critical error.
notumor_idx = class_names.index("notumor")
is_tumor = targets != notumor_idx
missed = int((preds[is_tumor] == notumor_idx).sum())
print(f"Tumor miss rate (true tumor predicted 'notumor'): "
      f"{missed}/{int(is_tumor.sum())} = {missed / is_tumor.sum():.4f}")

# Confusion matrix
cm = confusion_matrix(targets, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix (test set)")
plt.tight_layout(); plt.show()